## Regresión Logistica

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 1. Cargar los datos
# Reemplazar por la ruta de tu archivo CSV (ej: pd.read_csv('pacientes.csv'))
df = pd.read_csv("../data/corazon.csv")

# 2. Separar características (X) y objetivo (y)
X = df[["edad", "genero", "presion", "colesterol", "diabetico"]]
y = df["cardiaco"]

# 3. Dividir en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Definir qué columnas escalar y cuáles dejar sin cambios
cols_continuas = ["edad", "presion", "colesterol"]
cols_binarias = ["genero", "diabetico"]

preprocesador = ColumnTransformer(
    transformers=[
        ("scaler", StandardScaler(), cols_continuas),
        ("passthrough", "passthrough", cols_binarias),
    ]
)

# 5. Crear el Pipeline (une el preprocesamiento con el modelo)
modelo_pipeline = Pipeline(
    steps=[
        ("preprocesamiento", preprocesador),
        ("clasificador", LogisticRegression(random_state=42)),
    ]
)

# 6. Entrenar el pipeline
modelo_pipeline.fit(X_train, y_train)

# 7. Evaluar el modelo en el conjunto de prueba
y_pred = modelo_pipeline.predict(X_test)

print("--- Matriz de Confusión ---")
print(confusion_matrix(y_test, y_pred))

print("\n--- Reporte de Clasificación ---")
print(classification_report(y_test, y_pred))

# 8. (Opcional) Ver la importancia / peso de cada variable
# Se obtienen los nombres en el orden procesado por ColumnTransformer
features_ordenadas = cols_continuas + cols_binarias
coeficientes = modelo_pipeline.named_steps["clasificador"].coef_[0]

print("\n--- Coeficientes de la Regresión Logística ---")
for feat, coef in zip(features_ordenadas, coeficientes):
    print(f"{feat:12s}: {coef:+.4f} (Odds Ratio: {np.exp(coef):.4f})")

--- Matriz de Confusión ---
[[7 3]
 [4 6]]

--- Reporte de Clasificación ---
              precision    recall  f1-score   support

           0       0.64      0.70      0.67        10
           1       0.67      0.60      0.63        10

    accuracy                           0.65        20
   macro avg       0.65      0.65      0.65        20
weighted avg       0.65      0.65      0.65        20


--- Coeficientes de la Regresión Logística ---
edad        : -0.7156 (Odds Ratio: 0.4889)
presion     : -0.6717 (Odds Ratio: 0.5109)
colesterol  : +0.3519 (Odds Ratio: 1.4217)
genero      : -1.1162 (Odds Ratio: 0.3275)
diabetico   : +0.9247 (Odds Ratio: 2.5211)
